<a href="https://colab.research.google.com/github/sabthe02/Obligatorio-IA/blob/Pato/obligatorio_hotel_cancelaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Obligatorio: Predicción de cancelaciones de reservas hoteleras
**Electiva Inteligencia Artificial – Licenciatura en Informática – UDE**

**Integrantes:** _Sabine Theiss, Jorge Avellanal, Mateo Mendiondo_

**Dataset:** Hotel Booking Demand (Antonio, Almeida & Nunes, 2019, *Data in Brief* vol. 22, DOI 10.1016/j.dib.2018.11.126).
Versión usada: Kaggle `jessemostipak/hotel-booking-demand`
**Licencia:** _Attribution 4.0 International https://creativecommons.org/licenses/by/4.0/_

> Este notebook debe poder ejecutarse de punta a punta (Restart & Run All), con semillas fijas y sin celdas con error.

## 0. Setup y reproducibilidad

In [ ]:
import random


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

**requirements.txt:** la celda siguiente lo genera sin versiones. Antes de entregar, fijen las versiones reales con
`!pip freeze | grep -i -E "pandas|numpy|scikit-learn|matplotlib|seaborn|shap|xgboost|lightgbm|joblib"`
y copien el resultado al archivo.

In [ ]:
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

if importlib.util.find_spec("docx") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])

REQUIRED_DISTRIBUTIONS = [
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "joblib",
    "python-docx",
]
missing_distributions = []
requirements_lines = []
for distribution in REQUIRED_DISTRIBUTIONS:
    try:
        requirements_lines.append(f"{distribution}=={version(distribution)}")
    except PackageNotFoundError:
        missing_distributions.append(distribution)

if missing_distributions:
    raise RuntimeError(
        "Instala estas dependencias antes de generar requirements.txt y el informe DOCX: "
        + ", ".join(missing_distributions)
        + ". En Colab puedes ejecutar %pip install "
        + " ".join(missing_distributions)
        + " y volver a correr esta celda."
    )

Path("requirements.txt").write_text(
    "\n".join(requirements_lines) + "\n",
    encoding="utf-8",
)
print("requirements.txt generado con versiones instaladas:")
print("\n".join(requirements_lines))

Overwriting requirements.txt


## 1. Definición del problema

_Propuesta inicial para validar con el hotel/equipo antes de modelar._

- **Qué se predice:** si una reserva será cancelada (`is_canceled`), usando solo información disponible **al momento de reservar**.
- **Para qué y quién la usa:** el área de revenue management / reservas, para decidir cuánto sobrevender (overbooking) y a qué reservas hacer seguimiento.
- **Costo de los errores:**
  - Falso negativo (la reserva se cancela y el modelo no lo anticipó): podría dejar una habitación vacía y perderse la oportunidad de reasignarla.
  - Falso positivo (el modelo anticipa cancelación y la reserva se cumple): podría motivar una intervención innecesaria o una decisión de sobreventa. Sin conocer la acción ni sus consecuencias, no se puede afirmar cuál error es más costoso.
- **Proxy académico orientado a la operación (no es una métrica monetaria de negocio):** evaluar escenarios hipotéticos que marcan el 5%, 10% y 20% de reservas con mayor riesgo. Para cada escenario, informar recall@k (proporción de cancelaciones capturadas), precision@k (proporción de cancelaciones entre las alertas) y alertas por cada 1.000 reservas; comparar también con una selección aleatoria. Estos porcentajes son escenarios ilustrativos, no capacidades acordadas, y no implican que una intervención evite cancelaciones o genere ahorro.
- **Métrica técnica principal:** Average Precision para comparar la calidad del ranking en validación temporal dentro de train. Complementar con precision@k, recall@k, alertas por 1.000, ROC-AUC y calibración. No elegir un umbral monetario ni afirmar costos relativos sin datos; evaluar el test reservado una sola vez, después de cerrar el modelo y el análisis.
- **Descripción del dataset:** Hotel Booking Demand (Antonio, Almeida y Nunes, 2019): 119.390 reservas y 32 columnas; `is_canceled` indica si la reserva terminó cancelada (44.224 casos, 37,0%). Incluye dos hoteles de Portugal y reservas de julio de 2015 a agosto de 2017. Es observacional e histórico; no representa necesariamente otros hoteles/épocas, no contiene costos ni capacidad operativa, y algunas columnas describen eventos posteriores a la reserva y deben excluirse para evitar fuga.

## 2. Carga de datos y partición (antes del EDA)

La partición se define **antes** de explorar y modelar. El conjunto de test se guarda en un archivo y se elimina de la memoria:
no se vuelve a abrir hasta la evaluación final (sección 7).

In [ ]:
LOCAL_DATA_PATH = Path("hotel_bookings.csv")
URL = "https://raw.githubusercontent.com/sabthe02/Obligatorio-IA/Pato/hotel_bookings.csv"
if LOCAL_DATA_PATH.exists():
    df_raw = pd.read_csv(LOCAL_DATA_PATH)
    print(f"Dataset cargado desde el repositorio local: {LOCAL_DATA_PATH}")
else:
    df_raw = pd.read_csv(URL)
    print(f"Dataset cargado desde la rama Pato: {URL}")
print("Filas y columnas:", df_raw.shape)
df_raw.head()

Filas y columnas: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [ ]:
MESES = {m: i for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June",
     "July", "August", "September", "October", "November", "December"], start=1)}

df = df_raw.copy()
df["arrival_date"] = pd.to_datetime(dict(
    year=df["arrival_date_year"],
    month=df["arrival_date_month"].map(MESES),
    day=df["arrival_date_day_of_month"],
))
assert df["arrival_date"].notna().all(), "Hay fechas de llegada que no se pudieron construir"

print("Rango de fechas de llegada:", df["arrival_date"].min().date(), "->", df["arrival_date"].max().date())

Rango de fechas de llegada: 2015-07-01 -> 2017-08-31


In [ ]:
TARGET = "is_canceled"
TEST_FRAC = 0.20

# La fecha de reserva no está disponible; se aproxima restando lead_time a la llegada.
df["booking_date_est"] = df["arrival_date"] - pd.to_timedelta(df["lead_time"], unit="D")
df = df.sort_values("booking_date_est", kind="mergesort").reset_index(drop=True)
cut_idx = int(len(df) * (1 - TEST_FRAC))
booking_cutoff = df.loc[cut_idx, "booking_date_est"]
train_mask = (df["booking_date_est"] < booking_cutoff) & (df["arrival_date"] < booking_cutoff)
test_mask = df["booking_date_est"] >= booking_cutoff
purged_mask = (df["booking_date_est"] < booking_cutoff) & (df["arrival_date"] >= booking_cutoff)

train_df = df[train_mask].copy()
test_df = df[test_mask].copy()
purged_count = int(purged_mask.sum())

print(f"Corte por fecha estimada de reserva: {booking_cutoff.date()}")
print(f"Train maduro: {len(train_df):,} reservas; fechas de reserva {train_df['booking_date_est'].min().date()} -> {train_df['booking_date_est'].max().date()}")
print(f"Test futuro: {len(test_df):,} reservas; fechas de reserva {test_df['booking_date_est'].min().date()} -> {test_df['booking_date_est'].max().date()}")
print(f"Train excluido por llegada/resultado posterior al corte: {purged_count:,} reservas")
print("Proporción de cancelaciones en train:", f"{train_df[TARGET].mean():.1%}")

# booking_date_est solo se usa para separar el tiempo, no como predictor.
test_df.to_csv("test_holdout.csv", index=False)
del test_df, df, df_raw, train_mask, test_mask, purged_mask

**Decisiones y límites de la partición (para justificar en el informe y en la defensa):**
- El objetivo es predecir al crear la reserva. Como no hay fecha de creación, se aproxima por `booking_date_est = arrival_date - lead_time`; no representa necesariamente la fecha real exacta.
- El 20% más reciente por fecha estimada de reserva se reserva para test. El train incluye solo reservas estimadas antes del corte cuya llegada también ocurrió antes del corte; así sus resultados habrían madurado al momento simulado de entrenamiento. Las reservas anteriores al corte que llegan después se purgan. El test incluye reservas posteriores al corte con resultados finales disponibles en este dataset histórico.
- La selección por `booking_date_est` no implica causalidad ni elimina sesgos de supervivencia/registro. El test debe evaluarse una sola vez, después de fijar modelo y umbral; una versión anterior inspeccionó etiquetas de la partición por llegada, por lo que el conjunto no es completamente ciego.
- Duplicados: revisar cuántos exactos quedan en el nuevo train. No eliminarlos automáticamente: no hay un ID que permita distinguir duplicados accidentales de reservas independientes con atributos idénticos. Comparar deduplicación exacta como análisis de sensibilidad y evitar que grupos duplicados se repartan entre folds.
- Para validación cruzada, usar cortes temporales basados en `booking_date_est`, aplicando también la regla de madurez (`arrival_date` anterior al corte de cada fold).

## 3. EDA y calidad de datos (solo sobre train)

Checklist:
- [ ] Distribución del target y balance de clases (implicancia para métricas y manejo del desbalance)
- [ ] Valores faltantes y qué patrón siguen
- [ ] Outliers (por ejemplo `adr`, `adults`, `lead_time`, estadías de 0 noches)
- [ ] Duplicados
- [ ] Relación de las variables con el target
- [ ] Al menos 6 visualizaciones, cada una con su conclusión escrita
- [ ] Hipótesis que se llevan al modelado

In [ ]:
print("Distribución del target en train:")
print(train_df[TARGET].value_counts(normalize=True).rename("proporcion"))

nulls = train_df.isna().mean().sort_values(ascending=False)
print("\nVariables con valores faltantes:")
display(nulls[nulls > 0].to_frame("prop_nulos"))

print("Duplicados exactos en train:", train_df.duplicated().sum())

**Visualización 1:** proporción de reservas canceladas y no canceladas en train; permite dimensionar el desbalance antes de elegir métricas.

In [ ]:
target_share = train_df[TARGET].value_counts(normalize=True).sort_index()
ax = target_share.plot.bar(color=["#4C78A8", "#F58518"], rot=0)
ax.set_xticklabels(["No cancelada", "Cancelada"])
ax.set_ylabel("Proporción")
ax.set_title("Distribución del target en train")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
plt.tight_layout()
plt.show()

**Conclusión:** en el train temporal, el 36,1% de las reservas se canceló y el 63,9% no. La clase positiva es minoritaria, por lo que accuracy sola puede resultar engañosa; se priorizan Average Precision y el análisis conjunto de precisión/recall.

**Visualización 2:** tasa de cancelación por tipo de hotel, para comprobar si el contexto del hotel aporta señal predictiva.

In [ ]:
hotel_rates = train_df.groupby("hotel", observed=True)[TARGET].mean().sort_values()
ax = hotel_rates.plot.bar(color="#4C78A8", rot=0)
ax.set_ylabel("Tasa de cancelación")
ax.set_title("Cancelación por tipo de hotel (train)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
plt.tight_layout()
plt.show()

**Conclusión:** la tasa es aproximadamente 41,3% en City Hotel y 26,2% en Resort Hotel. La diferencia justifica probar `hotel` y medir su aporte en validación; es una asociación, no evidencia causal.

**Visualización 3:** tasa de cancelación por mes de llegada, para explorar estacionalidad.

In [ ]:
month_rates = train_df.groupby(train_df["arrival_date"].dt.month)[TARGET].mean()
ax = month_rates.plot(marker="o", color="#54A24B")
ax.set_xticks(range(1, 13))
ax.set_xlabel("Mes de llegada")
ax.set_ylabel("Tasa de cancelación")
ax.set_title("Cancelación por mes de llegada (train)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
plt.tight_layout()
plt.show()

**Conclusión:** las tasas mensuales varían aproximadamente de 27,2% (enero) a 39,6% (junio). Esto sugiere posible estacionalidad, que debe validarse temporalmente y no interpretarse como un efecto causal.

**Visualización 4:** tasa de cancelación por tramos de anticipación (`lead_time`), para evaluar una relación no lineal.

In [ ]:
lead_bins = [-1, 7, 30, 90, 180, 365, np.inf]
lead_labels = ["0-7", "8-30", "31-90", "91-180", "181-365", "366+"]
lead_band = pd.cut(train_df["lead_time"], bins=lead_bins, labels=lead_labels)
lead_rates = train_df.groupby(lead_band, observed=True)[TARGET].mean()
ax = lead_rates.plot.bar(color="#E45756", rot=30)
ax.set_xlabel("Días entre reserva y llegada")
ax.set_ylabel("Tasa de cancelación")
ax.set_title("Cancelación según anticipación (train)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
plt.tight_layout()
plt.show()

**Conclusión:** la tasa aumenta desde 9,9% para anticipaciones de hasta 7 días hasta 77,8% para más de 365 días (este último tramo tiene 1.801 reservas). `lead_time` parece una señal importante; conviene permitir relaciones no lineales y revisar calibración por tramo.

**Visualización 5:** tasa de cancelación por tipo de depósito, para evaluar si las condiciones de pago se asocian con la cancelación.

In [ ]:
deposit_rates = train_df.groupby("deposit_type", observed=True)[TARGET].agg(["mean", "size"]).sort_values("mean")
ax = deposit_rates["mean"].plot.bar(color="#B279A2", rot=20)
ax.set_ylabel("Tasa de cancelación")
ax.set_title("Cancelación por tipo de depósito (train)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
for position, (rate, count) in enumerate(zip(deposit_rates["mean"], deposit_rates["size"])):
    ax.annotate(f"n={count:,}", (position, rate), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

**Conclusión:** las reservas Non Refund presentan una tasa cercana a 99,2%, frente a 26,6% en No Deposit y 7,4% en Refundable; el grupo Refundable es pequeño (135 casos). `deposit_type` es muy informativa, pero queda excluida del modelo principal hasta confirmar que se conozca al momento de predecir; asociación no implica causalidad.

**Visualización 6:** distribución de `adr` por resultado, ocultando puntos extremos y mostrando hasta el percentil 99 para que el centro sea legible.

In [ ]:
adr_plot = train_df[[TARGET, "adr"]].dropna().copy()
adr_upper = adr_plot["adr"].quantile(0.99)
adr_plot.boxplot(column="adr", by=TARGET, showfliers=False, grid=False)
plt.ylim(0, adr_upper)
plt.xlabel("Cancelada (0 = no, 1 = sí)")
plt.ylabel("ADR (recortado visualmente al percentil 99)")
plt.title("ADR según resultado de la reserva")
plt.suptitle("")
plt.tight_layout()
plt.show()

**Conclusión:** las medianas de ADR son parecidas (88,20 para no canceladas y 89,96 para canceladas), así que esta variable por sí sola separa poco en el centro de la distribución. El máximo llega a 5.400; investigar ese extremo y validar la disponibilidad de `adr` al reservar antes de usarla.

## 4. Prevención de fuga de datos y estrategia de validación

### 4.1 Tabla de auditoría de variables (BORRADOR)

Para cada variable: si está disponible **al momento de reservar**, si puede ser un proxy del target y qué decisión se toma.
Este borrador sale del significado de cada columna, no de los datos: **hay que validarlo con el EDA** (por ejemplo,
comparando la distribución de cada variable dudosa entre reservas canceladas y no canceladas) y ajustarlo.

In [ ]:
audit_rows = [
    # variable, disponible al reservar, riesgo de proxy, decisión, justificación
    ("hotel", "Sí", "Bajo", "Usar", "Tipo de hotel (resort o ciudad), conocido al reservar."),
    ("is_canceled", "-", "-", "Target", "Variable a predecir."),
    ("lead_time", "Sí", "Bajo", "Usar", "Días entre la reserva y la llegada; se conoce al reservar."),
    ("arrival_date_year", "Sí", "Bajo", "Transformar o descartar", "Con partición temporal el test cae en años/fechas no vistas; puede no generalizar."),
    ("arrival_date_month", "Sí", "Bajo", "Transformar", "Captura estacionalidad; codificar como número o de forma cíclica."),
    ("arrival_date_week_number", "Sí", "Bajo", "Evaluar", "Redundante con el mes; decidir en la selección de variables."),
    ("arrival_date_day_of_month", "Sí", "Bajo", "Evaluar", "Poco aporte esperado; medir en la ablation."),
    ("stays_in_weekend_nights", "Sí", "Bajo", "Usar", "Estadía planificada; verificar que no refleje cambios posteriores."),
    ("stays_in_week_nights", "Sí", "Bajo", "Usar", "Estadía planificada; verificar que no refleje cambios posteriores."),
    ("adults", "Sí", "Bajo", "Usar", "Composición de la reserva."),
    ("children", "Sí", "Bajo", "Usar", "Composición de la reserva; tiene pocos faltantes."),
    ("babies", "Sí", "Bajo", "Usar", "Composición de la reserva."),
    ("meal", "Sí", "Bajo", "Usar", "Régimen de comidas contratado."),
    ("country", "Sí", "Bajo", "Transformar", "Alta cardinalidad y faltantes: frecuencia o agrupación de categorías raras."),
    ("market_segment", "Sí", "Bajo", "Usar", "Segmento de mercado; revisar categorías raras o 'Undefined'."),
    ("distribution_channel", "Sí", "Bajo", "Usar", "Canal de distribución; revisar categorías raras o 'Undefined'."),
    ("is_repeated_guest", "Sí", "Bajo", "Usar", "Dato del huésped previo a la reserva."),
    ("previous_cancellations", "Sí", "Bajo", "Usar", "Historial del cliente anterior a esta reserva."),
    ("previous_bookings_not_canceled", "Sí", "Bajo", "Usar", "Historial del cliente anterior a esta reserva."),
    ("reserved_room_type", "Sí", "Bajo", "Usar", "Tipo de habitación reservada."),
    ("assigned_room_type", "Dudosa", "Alto", "Descartar (a validar)", "Se asigna cerca de la estadía y puede diferir de la reservada; candidata a fuga. Validar con el EDA."),
    ("booking_changes", "No", "Alto", "Descartar (a validar)", "Cuenta cambios posteriores a la creación de la reserva; no se conoce al reservar."),
    ("deposit_type", "Dudosa", "Medio", "Excluir del modelo principal", "La disponibilidad exacta al reservar no está confirmada; excluir hasta verificarla."),
    ("agent", "Sí", "Bajo", "Transformar", "Alta cardinalidad y faltantes: frecuencia, agrupación e indicador de faltante."),
    ("company", "Sí", "Bajo", "Transformar", "Muchos faltantes: indicador de presencia, agrupación."),
    ("days_in_waiting_list", "Dudosa", "Medio", "Excluir del modelo principal", "La disponibilidad al reservar no está confirmada; excluir hasta verificarla."),
    ("customer_type", "Sí", "Bajo", "Usar", "Tipo de cliente (contrato, grupo, transient)."),
    ("adr", "Dudosa", "Medio", "Excluir del modelo principal", "La disponibilidad y cálculo al reservar no están confirmados; excluir hasta verificarlo."),
    ("required_car_parking_spaces", "Dudosa", "Medio", "Excluir del modelo principal", "La disponibilidad al reservar no está confirmada; excluir hasta verificarla."),
    ("total_of_special_requests", "Dudosa", "Medio", "Excluir del modelo principal", "La disponibilidad al reservar no está confirmada; excluir hasta verificarla."),
    ("reservation_status", "No", "Muy alto", "Descartar", "Fuga directa: informa si la reserva fue cancelada, hizo check-out o no se presentó. Solo para el experimento de fuga."),
    ("reservation_status_date", "No", "Muy alto", "Descartar", "Fecha del último estado de la reserva; fuga. Solo para el experimento de fuga."),
]

audit = pd.DataFrame(audit_rows, columns=[
    "variable", "disponible_al_reservar", "riesgo_proxy", "decision", "justificacion"])

# Control de cobertura: toda columna del dataset debe estar auditada.
cols_dataset = set(train_df.columns) - {"arrival_date", "booking_date_est"}  # Fechas derivadas
sin_auditar = cols_dataset - set(audit["variable"])
sobran = set(audit["variable"]) - cols_dataset
print("Sin auditar:", sin_auditar or "ninguna")
print("En la tabla pero no en el dataset:", sobran or "ninguna")

audit

In [ ]:
# Columnas que se usan SOLO en el experimento de fuga (sección 6.4), nunca en el modelo real.
LEAK_COLS = ["reservation_status", "reservation_status_date"]

### 4.2 Preprocesamiento sin fuga
_(Imputación, escalado y codificación dentro de un `Pipeline` / `ColumnTransformer`, ajustados solo con datos de entrenamiento, también dentro de la validación cruzada. Cualquier transformación que aprenda de los datos —frecuencias, target encoding, medias— va dentro del Pipeline.)_

## 5. Feature engineering

Mínimo 8 variables nuevas, cada una con su hipótesis. **Ideas candidatas** (validar cuáles se usan):

| Variable | Hipótesis de por qué podría ayudar |
|---|---|
| `total_nights` | Las estadías largas podrían tener distinta probabilidad de cancelación. |
| `total_guests` | El tamaño del grupo puede relacionarse con la dificultad de organizar el viaje. |
| `has_children` | Las reservas con niños podrían ser más o menos propensas a cancelarse. |
| `arrival_month` (cíclica) | Estacionalidad de las cancelaciones. |
| `arrival_dow` / llegada en fin de semana | El día de llegada puede cambiar el comportamiento. |
| `lead_time` por tramos | Reservas hechas con mucha anticipación suelen cancelarse más. |
| frecuencia de `country` | Reemplaza una variable de alta cardinalidad por su frecuencia. |
| frecuencia de `agent` | Idem para agencias. |
| `prev_cancel_ratio` | Proporción de cancelaciones previas del cliente. |
| `adr` por persona | Precio relativo al tamaño de la reserva. |

Además: tratamiento justificado de categóricas, **ablation** por familia de variables, y al menos un método de selección de variables.

In [ ]:
# Ejemplos de variables SIN estado (no aprenden de los datos, se pueden calcular fuera del Pipeline).
def add_stateless_features(d):
    d = d.copy()
    d["total_nights"] = d["stays_in_weekend_nights"] + d["stays_in_week_nights"]
    d["total_guests"] = d["adults"] + d["children"].fillna(0) + d["babies"]
    d["has_children"] = ((d["children"].fillna(0) + d["babies"]) > 0).astype(int)
    d["arrival_month"] = d["arrival_date"].dt.month
    d["arrival_dow"] = d["arrival_date"].dt.dayofweek
    d["arrival_is_weekend"] = d["arrival_dow"].isin([5, 6]).astype(int)
    d["arrival_month_sin"] = np.sin(2 * np.pi * d["arrival_month"] / 12)
    d["arrival_month_cos"] = np.cos(2 * np.pi * d["arrival_month"] / 12)
    d["lead_time_band"] = pd.cut(
        d["lead_time"],
        bins=[-1, 7, 30, 90, 180, 365, np.inf],
        labels=["0-7", "8-30", "31-90", "91-180", "181-365", "366+"],
    ).astype("object")
    prior_bookings = d["previous_cancellations"] + d["previous_bookings_not_canceled"]
    d["prev_cancel_ratio"] = d["previous_cancellations"].div(prior_bookings.where(prior_bookings > 0)).fillna(0)
    d["adr_per_guest"] = d["adr"].div(d["total_guests"].where(d["total_guests"] > 0))
    return d

train_df = add_stateless_features(train_df)

print("Variables derivadas:", ["total_nights", "total_guests", "has_children", "arrival_month", "arrival_dow", "arrival_is_weekend", "arrival_month_sin", "arrival_month_cos", "lead_time_band", "prev_cancel_ratio", "adr_per_guest"])
print("Dimensiones de train con variables derivadas:", train_df.shape)

### 5.1 Sensibilidad a duplicados exactos

El análisis principal conserva todos los registros. Se prepara una variante deduplicada solo para comparar resultados; no reemplaza `train_df`. En el modelado, ambas variantes deben usar la misma validación y mantener juntas las filas idénticas para evitar que copias exactas queden a ambos lados de un fold.

In [ ]:
BOOKING_TIME_UNVERIFIED_COLS = ["deposit_type", "adr", "days_in_waiting_list", "required_car_parking_spaces", "total_of_special_requests"]
POST_BOOKING_COLS = ["booking_changes", "assigned_room_type"]
MODEL_EXCLUDE_COLS = ["arrival_date", "booking_date_est", "reservation_status", "reservation_status_date", *POST_BOOKING_COLS, *BOOKING_TIME_UNVERIFIED_COLS, "adr_per_guest"]
MODEL_FEATURE_COLS = [col for col in train_df.columns if col not in MODEL_EXCLUDE_COLS + [TARGET]]
assert not set(MODEL_FEATURE_COLS).intersection(MODEL_EXCLUDE_COLS + [TARGET])

train_df_dedup_sensitivity = train_df.drop_duplicates().copy()
duplicates_removed = len(train_df) - len(train_df_dedup_sensitivity)
assert not train_df_dedup_sensitivity.duplicated().any()

print(f"Train principal: {len(train_df):,} filas; tasa de cancelación {train_df[TARGET].mean():.2%}")
print(f"Variante deduplicada: {len(train_df_dedup_sensitivity):,} filas; tasa de cancelación {train_df_dedup_sensitivity[TARGET].mean():.2%}")
print(f"Filas exactas removidas para la sensibilidad: {duplicates_removed:,}")

## 6. Modelado

### 6.1 Baselines
- [x] Modelo trivial (`DummyClassifier`)
- [x] Regresión logística

### 6.2 Comparación de algoritmos (al menos 4 familias)
- [x] Lineal (regresión logística)
- [x] Árboles / Random Forest
- [x] Boosting (Gradient Boosting)
- [x] Kernel SVM con aproximación RBF de Nyström

### 6.3 Validación y ajuste
- [x] Validación temporal purgada de 5 folds (dentro del train)
- [x] Búsqueda anidada y limitada para Random Forest y Gradient Boosting; ajuste solo dentro de train, nunca sobre el test
- [x] Selección de configuración en train: Gradient Boosting como candidato principal y Random Forest como benchmark; ambos ajustados con todo train
- [x] Manejo del desbalance: Average Precision y pesos de clase en modelos seleccionados
- [x] Calibración sigmoid implementada con folds temporales de train; falta ejecutar y evaluar su calidad
- [x] Escenarios hipotéticos de capacidad: top 5%, 10% y 20% de riesgo; informar precision@k, recall@k y alertas por 1.000 reservas, frente a selección aleatoria
- [x] Sensibilidad a duplicados: entrenar con `train_df` y `train_df_dedup_sensitivity` sobre las mismas filas de validación; copias exactas no cruzan folds

### 6.4 Experimento de fuga
- [x] Implementado: comparar Gradient Boosting honesto con una variante que agrega `reservation_status` y `reservation_status_date`, usando los mismos folds temporales de train
- [ ] Ejecutar el experimento y reportar la inflación observada de Average Precision; no usar el test

In [ ]:
def capacity_scenario_metrics(y_true, y_score, fractions=(0.05, 0.10, 0.20)):
    from sklearn.metrics import average_precision_score

    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score, dtype=float)
    if y_true.ndim != 1 or y_score.ndim != 1 or len(y_true) != len(y_score) or len(y_true) == 0:
        raise ValueError("y_true and y_score must be non-empty one-dimensional arrays of equal length")
    if not np.isin(y_true, [0, 1]).all() or not np.isfinite(y_score).all():
        raise ValueError("y_true must contain only 0/1 labels and y_score must be finite")
    positive_count = int(y_true.sum())
    if positive_count == 0:
        raise ValueError("At least one positive label is required to calculate recall")

    fractions = tuple(float(fraction) for fraction in fractions)
    if not fractions or any(not np.isfinite(fraction) or fraction <= 0 or fraction > 1 for fraction in fractions):
        raise ValueError("fractions must contain values in the interval (0, 1]")

    order = np.argsort(-y_score, kind="mergesort")
    rows = []
    for fraction in fractions:
        k = min(len(y_true), max(1, int(np.ceil(fraction * len(y_true)))))
        selected_positives = int(y_true[order[:k]].sum())
        rows.append({
            "fraction_alerted": fraction,
            "alerts_per_1000": k / len(y_true) * 1000,
            "precision_at_k": selected_positives / k,
            "recall_at_k": selected_positives / positive_count,
            "random_precision_reference": positive_count / len(y_true),
            "random_recall_reference": k / len(y_true),
        })
    return average_precision_score(y_true, y_score), pd.DataFrame(rows)

def make_purged_rolling_folds(data, n_splits=5, initial_train_fraction=0.5, validation_windows=None):
    if n_splits < 1 or not 0 < initial_train_fraction < 1:
        raise ValueError("n_splits must be positive and initial_train_fraction must be between 0 and 1")
    if data[["booking_date_est", "arrival_date", TARGET]].isna().any().any():
        raise ValueError("booking_date_est, arrival_date, and target must not contain missing values")

    if validation_windows is None:
        unique_dates, date_counts = np.unique(data["booking_date_est"].to_numpy(), return_counts=True)
        if len(unique_dates) <= n_splits:
            raise ValueError("Not enough distinct booking dates for the requested folds")
        first_validation = int(np.searchsorted(np.cumsum(date_counts), np.ceil(initial_train_fraction * len(data)), side="left") + 1)
        first_validation = min(max(first_validation, 1), len(unique_dates) - n_splits)
        remaining_dates = unique_dates[first_validation:]
        remaining_counts = date_counts[first_validation:]
        cumulative_counts = np.cumsum(remaining_counts)
        targets = np.arange(1, n_splits) * cumulative_counts[-1] / n_splits
        split_positions = np.searchsorted(cumulative_counts, targets, side="left") + 1
        if len(np.unique(split_positions)) != n_splits - 1:
            raise ValueError("Not enough distinct dates to create non-empty, approximately balanced validation folds")
        date_blocks = np.split(remaining_dates, split_positions)
        validation_windows = [(pd.Timestamp(block[0]), pd.Timestamp(block[-1])) for block in date_blocks]
    else:
        validation_windows = [(pd.Timestamp(start), pd.Timestamp(end)) for start, end in validation_windows]
        if len(validation_windows) != n_splits:
            raise ValueError("validation_windows must contain exactly n_splits date ranges")

    folds = []
    summary = []
    booking_dates = data["booking_date_est"]
    arrival_dates = data["arrival_date"]
    for fold_number, (start, end) in enumerate(validation_windows, start=1):
        if start > end:
            raise ValueError("Each validation window must start on or before it ends")
        train_mask = booking_dates.lt(start) & arrival_dates.lt(start)
        validation_mask = booking_dates.between(start, end, inclusive="both")
        purged_mask = booking_dates.lt(start) & arrival_dates.ge(start)
        train_indices = np.flatnonzero(train_mask.to_numpy())
        validation_indices = np.flatnonzero(validation_mask.to_numpy())
        if not len(train_indices) or not len(validation_indices):
            raise ValueError(f"Fold {fold_number} has an empty training or validation partition")
        if data.iloc[validation_indices][TARGET].sum() == 0:
            raise ValueError(f"Fold {fold_number} has no positive labels for Average Precision/recall")

        train_hashes = pd.util.hash_pandas_object(data.iloc[train_indices], index=False).to_numpy()
        validation_hashes = pd.util.hash_pandas_object(data.iloc[validation_indices], index=False).to_numpy()
        if np.intersect1d(train_hashes, validation_hashes).size:
            raise AssertionError("Exact duplicate rows must not cross a fold boundary")

        folds.append((train_indices, validation_indices))
        summary.append({
            "fold": fold_number,
            "validation_start": start,
            "validation_end": end,
            "train_rows": len(train_indices),
            "validation_rows": len(validation_indices),
            "purged_early_bookings": int(purged_mask.sum()),
        })
    return folds, pd.DataFrame(summary), validation_windows

# Generate windows once on the primary train set; reuse them for the deduplicated sensitivity run.
temporal_folds, temporal_fold_summary, temporal_validation_windows = make_purged_rolling_folds(train_df)
temporal_fold_summary

def build_model_suite(numeric_features, categorical_features):
    from sklearn.compose import ColumnTransformer
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.kernel_approximation import Nystroem
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.svm import LinearSVC

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([
                ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                ("scaler", StandardScaler(with_mean=False)),
            ]), numeric_features),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent", add_indicator=True)),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), categorical_features),
        ],
        sparse_threshold=0.3,
    )

    def make_pipeline(classifier, approximate_rbf=False):
        steps = [("preprocess", preprocessor)]
        if approximate_rbf:
            steps.append(("rbf_approximation", Nystroem(kernel="rbf", n_components=200, random_state=SEED)))
        steps.append(("classifier", classifier))
        return Pipeline(steps)

    return {
        "Dummy baseline": DummyClassifier(strategy="prior"),
        "Logistic regression": make_pipeline(LogisticRegression(
            max_iter=3000, solver="liblinear", tol=1e-3,
            class_weight="balanced", random_state=SEED)),
        "Random forest": make_pipeline(RandomForestClassifier(
            n_estimators=100, max_features="sqrt", min_samples_leaf=2,
            class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)),
        "Gradient boosting": make_pipeline(GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.05, max_depth=2,
            min_samples_leaf=20, subsample=0.8, random_state=SEED)),
        "RBF Nyström + linear SVM (approximation)": make_pipeline(
            LinearSVC(class_weight="balanced", dual="auto", max_iter=3000, random_state=SEED),
            approximate_rbf=True),
    }

def evaluate_temporal_model_suite(data, folds, models, analysis_name, validation_data=None, validation_folds=None):
    from sklearn.base import clone
    from sklearn.dummy import DummyClassifier
    from sklearn.metrics import average_precision_score

    validation_data = data if validation_data is None else validation_data
    validation_folds = folds if validation_folds is None else validation_folds
    if len(folds) != len(validation_folds):
        raise ValueError("Training and validation fold lists must have equal length")
    X_train_data = data[MODEL_FEATURE_COLS]
    y_train_data = data[TARGET].to_numpy(dtype=int)
    X_validation_data = validation_data[MODEL_FEATURE_COLS]
    y_validation_data = validation_data[TARGET].to_numpy(dtype=int)
    fold_rows = []
    capacity_rows = []
    for model_name, model in models.items():
        for fold_number, ((train_indices, _), (_, validation_indices)) in enumerate(zip(folds, validation_folds), start=1):
            X_train = X_train_data.iloc[train_indices]
            X_validation = X_validation_data.iloc[validation_indices]
            y_train = y_train_data[train_indices]
            y_validation = y_validation_data[validation_indices]
            fitted_model = clone(model)
            if isinstance(fitted_model, DummyClassifier):
                fitted_model.fit(np.zeros((len(train_indices), 1)), y_train)
                scores = fitted_model.predict_proba(np.zeros((len(validation_indices), 1)))[:, 1]
            else:
                fitted_model.fit(X_train, y_train)
                if hasattr(fitted_model, "predict_proba"):
                    scores = fitted_model.predict_proba(X_validation)[:, 1]
                else:
                    scores = fitted_model.decision_function(X_validation)

            fold_rows.append({
                "analysis": analysis_name,
                "model": model_name,
                "fold": fold_number,
                "average_precision": average_precision_score(y_validation, scores),
                "random_baseline_average_precision": y_validation.mean(),
                "train_rows": len(train_indices),
                "validation_rows": len(validation_indices),
            })
            if not isinstance(fitted_model, DummyClassifier):
                _, scenarios = capacity_scenario_metrics(y_validation, scores)
                scenarios["analysis"] = analysis_name
                scenarios["model"] = model_name
                scenarios["fold"] = fold_number
                capacity_rows.append(scenarios)

    return pd.DataFrame(fold_rows), pd.concat(capacity_rows, ignore_index=True)

model_input = train_df[MODEL_FEATURE_COLS]
categorical_features = model_input.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_features = model_input.select_dtypes(include=[np.number, "bool"]).columns.tolist()
unhandled_features = set(model_input.columns) - set(categorical_features) - set(numeric_features)
if unhandled_features:
    raise TypeError(f"Add preprocessing for unsupported feature types: {sorted(unhandled_features)}")
model_suite = build_model_suite(numeric_features, categorical_features)
print("Model families ready:", list(model_suite))
# For dedup sensitivity, use dedup folds for training and primary folds for identical validation rows. Never pass the reserved test.

### 6.5 Comparación inicial en validación temporal

Esta celda ejecuta 5 modelos en 5 folds purgados para el train principal y repite el ajuste con train deduplicado, manteniendo las mismas filas de validación en ambas comparaciones. Todo el preprocesamiento se ajusta dentro de cada fold. Puede tardar varios minutos; no carga ni usa el holdout final. Los resultados son una comparación inicial, no una evaluación final ni una búsqueda de hiperparámetros.

In [ ]:
deduplicated_temporal_folds, deduplicated_fold_summary, _ = make_purged_rolling_folds(
    train_df_dedup_sensitivity,
    validation_windows=temporal_validation_windows,
)

primary_cv_results, primary_capacity_results = evaluate_temporal_model_suite(
    train_df, temporal_folds, model_suite, "primary")
deduplicated_train_cv_results, deduplicated_train_capacity_results = evaluate_temporal_model_suite(
    train_df_dedup_sensitivity,
    deduplicated_temporal_folds,
    model_suite,
    "deduplicated_train",
    validation_data=train_df,
    validation_folds=temporal_folds,
)

cv_results = pd.concat([primary_cv_results, deduplicated_train_cv_results], ignore_index=True)
cv_summary = cv_results.groupby(["analysis", "model"], as_index=False).agg(
    average_precision_mean=("average_precision", "mean"),
    average_precision_std=("average_precision", "std"),
    random_baseline_average_precision=("random_baseline_average_precision", "mean"),
)
display(cv_summary.sort_values(["analysis", "average_precision_mean"], ascending=[True, False]))

capacity_results = pd.concat(
    [primary_capacity_results, deduplicated_train_capacity_results], ignore_index=True)
capacity_summary = capacity_results.groupby(
    ["analysis", "model", "fraction_alerted"], as_index=False).agg(
        precision_at_k=("precision_at_k", "mean"),
        recall_at_k=("recall_at_k", "mean"),
        alerts_per_1000=("alerts_per_1000", "mean"),
        random_precision_reference=("random_precision_reference", "mean"),
        random_recall_reference=("random_recall_reference", "mean"),
    )
display(capacity_summary)

### 6.6 Búsqueda anidada para los dos candidatos principales

La búsqueda compara una cuadrícula pequeña de Random Forest y Gradient Boosting dentro de cada fold externo. Los hiperparámetros se eligen solo con 3 folds internos purgados; luego se ajusta cada candidato elegido en el entrenamiento externo y se evalúa en las mismas filas de validación principal. La variante deduplicada cambia solo el entrenamiento, reutilizando el modelo/hiperparámetros elegidos y el mismo fold externo. Esta celda requiere más tiempo de cómputo; nunca usa el test.

In [ ]:
TUNING_CANDIDATES = {
    "Random forest": [
        {"classifier__n_estimators": 100, "classifier__max_features": "sqrt", "classifier__min_samples_leaf": 2},
        {"classifier__n_estimators": 100, "classifier__max_features": "sqrt", "classifier__min_samples_leaf": 5},
        {"classifier__n_estimators": 100, "classifier__max_features": 0.5, "classifier__min_samples_leaf": 2},
    ],
    "Gradient boosting": [
        {"classifier__n_estimators": 100, "classifier__learning_rate": 0.05, "classifier__max_depth": 2},
        {"classifier__n_estimators": 150, "classifier__learning_rate": 0.05, "classifier__max_depth": 2},
        {"classifier__n_estimators": 100, "classifier__learning_rate": 0.1, "classifier__max_depth": 2},
    ],
}

def run_nested_temporal_search(primary_data, deduplicated_data, outer_folds,
                               deduplicated_outer_folds, validation_data, models,
                               n_inner_splits=3):
    from sklearn.base import clone

    if len(outer_folds) != len(deduplicated_outer_folds):
        raise ValueError("Primary and deduplicated data must have matching outer folds")
    candidate_rows = []
    outer_rows = []
    capacity_frames = []

    for outer_number, (primary_fold, deduplicated_fold) in enumerate(
            zip(outer_folds, deduplicated_outer_folds), start=1):
        primary_train_indices, validation_indices = primary_fold
        deduplicated_train_indices, _ = deduplicated_fold
        primary_outer_train = primary_data.iloc[primary_train_indices].copy()
        deduplicated_outer_train = deduplicated_data.iloc[deduplicated_train_indices].copy()
        outer_validation = validation_data.iloc[validation_indices].copy()
        inner_folds, _, _ = make_purged_rolling_folds(
            primary_outer_train,
            n_splits=n_inner_splits,
            initial_train_fraction=0.5,
        )

        for model_name, candidates in TUNING_CANDIDATES.items():
            model_scores = []
            for candidate_number, parameters in enumerate(candidates, start=1):
                candidate = clone(models[model_name])
                candidate.set_params(**parameters)
                inner_results, _ = evaluate_temporal_model_suite(
                    primary_outer_train,
                    inner_folds,
                    {model_name: candidate},
                    f"outer_{outer_number}_inner",
                )
                mean_inner_ap = inner_results["average_precision"].mean()
                model_scores.append((mean_inner_ap, candidate_number, parameters))
                candidate_rows.append({
                    "outer_fold": outer_number,
                    "model": model_name,
                    "candidate": candidate_number,
                    "parameters": parameters,
                    "inner_average_precision_mean": mean_inner_ap,
                    "inner_average_precision_std": inner_results["average_precision"].std(),
                })

            best_inner_ap, best_candidate_number, best_parameters = max(model_scores, key=lambda row: row[0])
            candidate_rows[-len(candidates) + best_candidate_number - 1]["selected"] = True
            chosen_model = clone(models[model_name])
            chosen_model.set_params(**best_parameters)

            primary_outer_fold = [(np.arange(len(primary_outer_train)), np.arange(len(outer_validation)))]
            primary_result, primary_capacity = evaluate_temporal_model_suite(
                primary_outer_train,
                primary_outer_fold,
                {model_name: chosen_model},
                "nested_primary",
                validation_data=outer_validation,
                validation_folds=primary_outer_fold,
            )
            deduplicated_outer_fold = [(np.arange(len(deduplicated_outer_train)), np.arange(len(outer_validation)))]
            deduplicated_result, deduplicated_capacity = evaluate_temporal_model_suite(
                deduplicated_outer_train,
                deduplicated_outer_fold,
                {model_name: chosen_model},
                "nested_deduplicated_train",
                validation_data=outer_validation,
                validation_folds=primary_outer_fold,
            )

            for analysis, result, capacity in [
                ("nested_primary", primary_result, primary_capacity),
                ("nested_deduplicated_train", deduplicated_result, deduplicated_capacity),
            ]:
                result_row = result.iloc[0].to_dict()
                result_row.update({
                    "analysis": analysis,
                    "outer_fold": outer_number,
                    "selected_candidate": best_candidate_number,
                    "selected_parameters": best_parameters,
                    "best_inner_average_precision": best_inner_ap,
                })
                outer_rows.append(result_row)
                capacity["outer_fold"] = outer_number
                capacity["selected_parameters"] = str(best_parameters)
                capacity_frames.append(capacity)

    return (pd.DataFrame(candidate_rows), pd.DataFrame(outer_rows),
            pd.concat(capacity_frames, ignore_index=True))

nested_candidate_results, nested_outer_results, nested_capacity_results = run_nested_temporal_search(
    train_df,
    train_df_dedup_sensitivity,
    temporal_folds,
    deduplicated_temporal_folds,
    train_df,
    model_suite,
)

nested_outer_summary = nested_outer_results.groupby(["analysis", "model"], as_index=False).agg(
    average_precision_mean=("average_precision", "mean"),
    average_precision_std=("average_precision", "std"),
)
display(nested_outer_summary)
display(nested_candidate_results[nested_candidate_results["selected"] == True])
display(nested_capacity_results.groupby(
    ["analysis", "model", "fraction_alerted"], as_index=False).agg(
        precision_at_k=("precision_at_k", "mean"),
        recall_at_k=("recall_at_k", "mean"),
        alerts_per_1000=("alerts_per_1000", "mean"),
    ))

### 6.7 Modelo candidato principal y benchmark

Por la decisi?n acordada, Gradient Boosting ser? el candidato principal y Random Forest se conservar? como benchmark. Para cada uno, se selecciona la configuraci?n con mayor Average Precision media en los folds temporales purgados de train; despu?s se ajustan ambos pipelines con todo train. Esta comparaci?n sirve para la selecci?n, no es una estimaci?n independiente del desempe?o. El modelo ganador se calibrar? con una log?stica temporal purgada (*sigmoid*) usando solo train; no se fija un umbral operativo ni se toca el test reservado.


In [ ]:
from sklearn.base import clone

final_training_rows = []
final_fitted_models = {}

for model_name in ("Gradient boosting", "Random forest"):
    candidate_scores = []
    for candidate_number, parameters in enumerate(TUNING_CANDIDATES[model_name], start=1):
        candidate = clone(model_suite[model_name])
        candidate.set_params(**parameters)
        fold_results, _ = evaluate_temporal_model_suite(
            train_df,
            temporal_folds,
            {model_name: candidate},
            f"final_train_selection_{model_name}",
        )
        mean_ap = float(fold_results["average_precision"].mean())
        std_ap = float(fold_results["average_precision"].std())
        candidate_scores.append((mean_ap, candidate_number, parameters))
        final_training_rows.append({
            "model": model_name,
            "candidate": candidate_number,
            "parameters": parameters,
            "average_precision_mean": mean_ap,
            "average_precision_std": std_ap,
        })

    best_mean_ap, best_candidate_number, best_parameters = max(
        candidate_scores, key=lambda row: row[0]
    )
    best_row = next(
        row for row in final_training_rows
        if row["model"] == model_name and row["candidate"] == best_candidate_number
    )
    best_row["selected"] = True

    final_model = clone(model_suite[model_name])
    final_model.set_params(**best_parameters)
    final_model.fit(train_df[MODEL_FEATURE_COLS], train_df[TARGET].to_numpy(dtype=int))
    final_fitted_models[model_name] = final_model

    print(
        f"{model_name}: candidato {best_candidate_number}; "
        f"AP media de selección={best_mean_ap:.4f}; parámetros={best_parameters}"
    )

final_training_selection = pd.DataFrame(final_training_rows)
display(final_training_selection.sort_values(
    ["model", "average_precision_mean"], ascending=[True, False]
))

# Calibraci?n del modelo ganador con validaci?n temporal purgada dentro de train.
from sklearn.calibration import CalibratedClassifierCV

primary_model_name = "Gradient boosting"
primary_selection = final_training_selection.loc[
    final_training_selection["model"] == primary_model_name
].query("selected == True").iloc[0]
primary_model = clone(model_suite[primary_model_name])
primary_model.set_params(**primary_selection["parameters"])
primary_model_calibrated = CalibratedClassifierCV(
    estimator=primary_model,
    method="sigmoid",
    cv=temporal_folds,
)
primary_model_calibrated.fit(
    train_df[MODEL_FEATURE_COLS],
    train_df[TARGET].to_numpy(dtype=int),
)
print("Calibraci?n sigmoid aplicada sobre folds temporales purgados de train. Test reservado permanece cerrado.")
final_model = primary_model_calibrated
final_model_name = primary_model_name


### 6.8 Calibraci?n de probabilidades

La calibraci?n se hace solo dentro de train, con las mismas ventanas temporales purgadas usadas para elegir el modelo. Se usa `sigmoid` porque es una forma conservadora de transformar scores en probabilidades sin abrir el test. En la etapa final solo se reportan resultados de train y, m?s adelante, se eval?a una sola vez el test reservado.

### Ablation y selección de variables

Se compara el Gradient Boosting con todas las variables contra variantes que retiran, una por vez, las familias de composición/estadía, calendario, anticipación e historial. Las hipótesis son que la composición y duración resumen el tamaño del viaje; el calendario captura estacionalidad; los tramos de anticipación permiten no linealidad; y el historial previo aporta señal del comportamiento del huésped. Se usan los mismos cinco folds temporales y una configuración fija seleccionada solo dentro de train, por lo que la ablation es un análisis de sensibilidad, no una estimación independiente de desempeño. La selección `SelectFromModel` con ExtraTrees se ajusta dentro del pipeline de cada fold; se reporta Average Precision y cuántas variables transformadas retiene.

In [ ]:
def evaluate_feature_set(data, folds, estimator, feature_columns, analysis_name):
    from sklearn.base import clone
    from sklearn.metrics import average_precision_score

    feature_columns = list(feature_columns)
    if not feature_columns:
        raise ValueError("At least one feature is required")

    X = data[feature_columns]
    y = data[TARGET].to_numpy(dtype=int)
    rows = []

    for fold_number, (train_indices, validation_indices) in enumerate(folds, start=1):
        fitted_model = clone(estimator)
        X_train = X.iloc[train_indices]
        X_validation = X.iloc[validation_indices]
        y_train = y[train_indices]
        y_validation = y[validation_indices]
        fitted_model.fit(X_train, y_train)
        if hasattr(fitted_model, "predict_proba"):
            scores = fitted_model.predict_proba(X_validation)[:, 1]
        else:
            scores = fitted_model.decision_function(X_validation)

        row = {
            "analysis": analysis_name,
            "fold": fold_number,
            "average_precision": average_precision_score(y_validation, scores),
            "n_features": len(feature_columns),
        }
        if "select" in getattr(fitted_model, "named_steps", {}):
            row["selected_transformed_features"] = int(
                fitted_model.named_steps["select"].get_support().sum()
            )
        rows.append(row)

    return pd.DataFrame(rows)


# Ablation: each feature family is removed in turn and evaluated on the same temporal folds.
ENGINEERED_FEATURE_GROUPS = {
    "composicion_y_estadia": ["total_nights", "total_guests", "has_children"],
    "calendario": [
        "arrival_month", "arrival_dow", "arrival_is_weekend",
        "arrival_month_sin", "arrival_month_cos",
    ],
    "anticipacion": ["lead_time_band"],
    "historial": ["prev_cancel_ratio"],
}

ablation_feature_sets = {"completo": MODEL_FEATURE_COLS.copy()}
for group_name, group_columns in ENGINEERED_FEATURE_GROUPS.items():
    ablation_feature_sets[f"sin_{group_name}"] = [
        column for column in MODEL_FEATURE_COLS if column not in group_columns
    ]

ablation_rows = []
for analysis_name, feature_columns in ablation_feature_sets.items():
    feature_frame = train_df[feature_columns]
    numeric_columns = feature_frame.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_columns = feature_frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    subset_suite = build_model_suite(numeric_columns, categorical_columns)
    ablation_model = clone(subset_suite["Gradient boosting"])
    ablation_model.set_params(**primary_selection["parameters"])
    ablation_rows.append(
        evaluate_feature_set(
            train_df, temporal_folds, ablation_model, feature_columns, analysis_name
        )
    )

ablation_results = pd.concat(ablation_rows, ignore_index=True)
ablation_summary = ablation_results.groupby("analysis", as_index=False).agg(
    average_precision_mean=("average_precision", "mean"),
    average_precision_std=("average_precision", "std"),
)
ablation_summary["delta_vs_full"] = (
    ablation_summary["average_precision_mean"]
    - ablation_summary.loc[ablation_summary["analysis"] == "completo", "average_precision_mean"].iloc[0]
)
display(ablation_summary.sort_values("average_precision_mean", ascending=False))


# Model-based selection is fitted inside each training fold after preprocessing.
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.base import clone

rf_preprocessor = clone(model_suite["Random forest"].named_steps["preprocess"])
feature_selection_model = Pipeline([
    ("preprocess", rf_preprocessor),
    ("select", SelectFromModel(
        ExtraTreesClassifier(
            n_estimators=100,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED,
        ),
        threshold="median",
    )),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED,
    )),
])

selection_results = evaluate_feature_set(
    train_df,
    temporal_folds,
    feature_selection_model,
    MODEL_FEATURE_COLS,
    "random_forest_select_from_model",
)
display(selection_results)
print(
    "La ablation y la selección son análisis internos sobre train; "
    "no constituyen una evaluación independiente ni usan el test reservado."
)

### Experimento controlado de fuga de datos

Se compara el Gradient Boosting honesto con una variante deliberadamente inválida que agrega `reservation_status` y `reservation_status_date`. Ambos usan los mismos cinco folds temporales de train, el mismo algoritmo y la configuración seleccionada solo dentro de train. El preprocesamiento se ajusta dentro de cada fold. Esta prueba cuantifica la inflación causada por información posterior al resultado; no se usa para seleccionar el modelo ni se consulta el test reservado.

In [ ]:
leaky_feature_columns = MODEL_FEATURE_COLS + LEAK_COLS
leaky_input = train_df[leaky_feature_columns]
leaky_categorical_features = leaky_input.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
leaky_numeric_features = leaky_input.select_dtypes(
    include=[np.number, "bool"]
).columns.tolist()
leaky_model_suite = build_model_suite(
    leaky_numeric_features, leaky_categorical_features
)

honest_model_for_leak_test = clone(model_suite["Gradient boosting"])
honest_model_for_leak_test.set_params(**primary_selection["parameters"])
leaky_model_for_test = clone(leaky_model_suite["Gradient boosting"])
leaky_model_for_test.set_params(**primary_selection["parameters"])

honest_leak_results = evaluate_feature_set(
    train_df,
    temporal_folds,
    honest_model_for_leak_test,
    MODEL_FEATURE_COLS,
    "honesto_sin_fuga",
)
leaky_results = evaluate_feature_set(
    train_df,
    temporal_folds,
    leaky_model_for_test,
    leaky_feature_columns,
    "deliberadamente_con_fuga",
)

leak_results = pd.concat([honest_leak_results, leaky_results], ignore_index=True)
leak_summary = leak_results.groupby("analysis", as_index=False).agg(
    average_precision_mean=("average_precision", "mean"),
    average_precision_std=("average_precision", "std"),
)
honest_ap = leak_summary.loc[
    leak_summary["analysis"] == "honesto_sin_fuga", "average_precision_mean"
].iloc[0]
leaky_ap = leak_summary.loc[
    leak_summary["analysis"] == "deliberadamente_con_fuga", "average_precision_mean"
].iloc[0]
leak_summary["delta_vs_honest"] = leak_summary["average_precision_mean"] - honest_ap
print(f"Cambio de AP por la fuga deliberada: {leaky_ap - honest_ap:+.4f}")
display(leak_summary)
display(leak_results)
print(
    "La variante con fuga es solo una demostración metodológica; "
    "no es un modelo válido ni debe evaluarse sobre el test."
)

### Calibración y sensibilidad al costo de errores

Los costos operativos reales no están disponibles, por lo que no se fija un umbral de negocio ni se inventan importes. La celda siguiente obtiene probabilidades calibradas fuera de muestra en folds temporales internos de train, evalúa su calibración con Brier score, log loss y curva de calibración, y muestra umbrales/costos normalizados bajo varios escenarios hipotéticos de costo relativo entre falso negativo y falso positivo. Estos escenarios sirven para discutir la decisión con la cátedra; no constituyen costos reales ni seleccionan automáticamente un umbral final.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss, log_loss

calibration_oof_rows = []
for outer_fold_number, (outer_train_indices, outer_validation_indices) in enumerate(
    temporal_folds, start=1
):
    outer_train = train_df.iloc[outer_train_indices].reset_index(drop=True)
    outer_validation = train_df.iloc[outer_validation_indices]
    inner_folds, _, _ = make_purged_rolling_folds(
        outer_train,
        n_splits=3,
        initial_train_fraction=0.5,
    )
    selected_candidate = nested_candidate_results.loc[
        (nested_candidate_results["outer_fold"] == outer_fold_number)
        & (nested_candidate_results["model"] == "Gradient boosting")
        & (nested_candidate_results["selected"] == True)
    ].iloc[0]
    fold_estimator = clone(model_suite["Gradient boosting"])
    fold_estimator.set_params(**selected_candidate["parameters"])
    fold_calibrated = CalibratedClassifierCV(
        estimator=fold_estimator,
        method="sigmoid",
        cv=inner_folds,
    )
    fold_calibrated.fit(
        outer_train[MODEL_FEATURE_COLS],
        outer_train[TARGET].to_numpy(dtype=int),
    )
    fold_probabilities = fold_calibrated.predict_proba(
        outer_validation[MODEL_FEATURE_COLS]
    )[:, 1]
    calibration_oof_rows.append(pd.DataFrame({
        "outer_fold": outer_fold_number,
        "y_true": outer_validation[TARGET].to_numpy(dtype=int),
        "y_probability": fold_probabilities,
    }))

calibration_oof = pd.concat(calibration_oof_rows, ignore_index=True)
calibration_metrics = pd.DataFrame([{
    "brier_score": brier_score_loss(
        calibration_oof["y_true"], calibration_oof["y_probability"]
    ),
    "log_loss": log_loss(
        calibration_oof["y_true"], calibration_oof["y_probability"], labels=[0, 1]
    ),
    "validation_rows": len(calibration_oof),
}])
display(calibration_metrics)

fraction_of_positives, mean_predicted_value = calibration_curve(
    calibration_oof["y_true"],
    calibration_oof["y_probability"],
    n_bins=10,
    strategy="quantile",
)
plt.plot(mean_predicted_value, fraction_of_positives, marker="o", label="Gradient Boosting calibrado")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibración perfecta")
plt.xlabel("Probabilidad media predicha")
plt.ylabel("Fracción observada de cancelaciones")
plt.title("Calibración temporal fuera de muestra (train)")
plt.legend()
plt.tight_layout()
plt.show()

cost_scenarios = [
    {"scenario": "FP=1, FN=1", "cost_fp": 1.0, "cost_fn": 1.0},
    {"scenario": "FP=1, FN=2", "cost_fp": 1.0, "cost_fn": 2.0},
    {"scenario": "FP=1, FN=5", "cost_fp": 1.0, "cost_fn": 5.0},
    {"scenario": "FP=1, FN=10", "cost_fp": 1.0, "cost_fn": 10.0},
    {"scenario": "FP=2, FN=1", "cost_fp": 2.0, "cost_fn": 1.0},
]
y_cost = calibration_oof["y_true"].to_numpy(dtype=int)
p_cost = calibration_oof["y_probability"].to_numpy(dtype=float)
cost_rows = []
for scenario in cost_scenarios:
    threshold = scenario["cost_fp"] / (scenario["cost_fp"] + scenario["cost_fn"])
    predicted_cancel = p_cost >= threshold
    false_positives = int(((predicted_cancel == 1) & (y_cost == 0)).sum())
    false_negatives = int(((predicted_cancel == 0) & (y_cost == 1)).sum())
    cost_rows.append({
        **scenario,
        "threshold": threshold,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "normalized_cost_per_reservation": (
            scenario["cost_fp"] * false_positives
            + scenario["cost_fn"] * false_negatives
        ) / len(y_cost),
        "alerts_per_1000": predicted_cancel.mean() * 1000,
    })
cost_sensitivity = pd.DataFrame(cost_rows)
display(cost_sensitivity)
print(
    "Los costos son unidades relativas hipotéticas. No elegir un escenario ni un umbral "
    "operativo hasta acordarlo con la cátedra o contar con costos del negocio."
)

## 7. Evaluación final e interpretación

La evaluación debe ejecutarse **una sola vez** sobre `test_holdout.csv`, después de cerrar modelo y umbral. El informe de avance advierte que una versión previa inspeccionó etiquetas de un holdout por fecha de llegada y que hay filas compartidas; por eso esta partición no es completamente ciega. No usar estos resultados para elegir hiperparámetros ni reentrenar. La celda está protegida: antes de ejecutarla, revisar la limitación, confirmar el umbral acordado con el equipo/cátedra y cambiar explícitamente `RUN_FINAL_TEST_EVALUATION` a `True`. La celda carga el test una sola vez y calcula todas las métricas y el bootstrap en el mismo bloque.

In [ ]:
RUN_FINAL_TEST_EVALUATION = False
FINAL_THRESHOLD = None  # Set only after agreeing on a cost scenario with the team/cátedra.

if not RUN_FINAL_TEST_EVALUATION:
    print(
        "Test cerrado. Fijar FINAL_THRESHOLD y confirmar la evaluación final antes de "
        "cambiar RUN_FINAL_TEST_EVALUATION a True."
    )
else:
    if FINAL_THRESHOLD is None or not 0 < float(FINAL_THRESHOLD) < 1:
        raise ValueError("Set FINAL_THRESHOLD to the agreed value in (0, 1) before opening test")

    from sklearn.metrics import (
        average_precision_score,
        brier_score_loss,
        classification_report,
        confusion_matrix,
        f1_score,
        log_loss,
        precision_recall_curve,
        precision_score,
        recall_score,
        roc_auc_score,
    )

    # This is the only cell that reads the reserved holdout.
    final_test_df = pd.read_csv("test_holdout.csv", parse_dates=["arrival_date"])
    final_test_df = add_stateless_features(final_test_df)
    X_test_final = final_test_df[MODEL_FEATURE_COLS]
    y_test_final = final_test_df[TARGET].to_numpy(dtype=int)
    test_probabilities = final_model.predict_proba(X_test_final)[:, 1]
    test_predictions = (test_probabilities >= float(FINAL_THRESHOLD)).astype(int)

    final_test_metrics = pd.DataFrame([{
        "average_precision": average_precision_score(y_test_final, test_probabilities),
        "roc_auc": roc_auc_score(y_test_final, test_probabilities),
        "precision": precision_score(y_test_final, test_predictions, zero_division=0),
        "recall": recall_score(y_test_final, test_predictions, zero_division=0),
        "f1": f1_score(y_test_final, test_predictions, zero_division=0),
        "brier_score": brier_score_loss(y_test_final, test_probabilities),
        "log_loss": log_loss(y_test_final, test_probabilities, labels=[0, 1]),
        "threshold": float(FINAL_THRESHOLD),
        "test_rows": len(y_test_final),
    }])
    display(final_test_metrics)
    print("Matriz de confusión (filas = real; columnas = predicho):")
    display(pd.DataFrame(
        confusion_matrix(y_test_final, test_predictions),
        index=["Real: no cancela", "Real: cancela"],
        columns=["Predicho: no cancela", "Predicho: cancela"],
    ))
    print(classification_report(
        y_test_final,
        test_predictions,
        target_names=["No cancelada", "Cancelada"],
        zero_division=0,
    ))

    test_average_precision, test_capacity_scenarios = capacity_scenario_metrics(
        y_test_final, test_probabilities
    )
    print(f"Average Precision en test: {test_average_precision:.4f}")
    display(test_capacity_scenarios)

    precision_points, recall_points, _ = precision_recall_curve(
        y_test_final, test_probabilities
    )
    plt.plot(recall_points, precision_points)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Curva precision-recall en el test reservado")
    plt.tight_layout()
    plt.show()

    bootstrap_rng = np.random.default_rng(SEED)
    bootstrap_rows = []
    bootstrap_replicates = 1000
    for _ in range(bootstrap_replicates):
        sample_indices = bootstrap_rng.integers(0, len(y_test_final), len(y_test_final))
        y_sample = y_test_final[sample_indices]
        probability_sample = test_probabilities[sample_indices]
        prediction_sample = test_predictions[sample_indices]
        if np.unique(y_sample).size < 2:
            continue
        bootstrap_rows.append({
            "average_precision": average_precision_score(y_sample, probability_sample),
            "roc_auc": roc_auc_score(y_sample, probability_sample),
            "precision": precision_score(y_sample, prediction_sample, zero_division=0),
            "recall": recall_score(y_sample, prediction_sample, zero_division=0),
            "f1": f1_score(y_sample, prediction_sample, zero_division=0),
        })

    bootstrap_frame = pd.DataFrame(bootstrap_rows)
    confidence_intervals = pd.DataFrame([
        {
            "metric": metric,
            "estimate": final_test_metrics.loc[0, metric],
            "ci_2_5%": bootstrap_frame[metric].quantile(0.025),
            "ci_97_5%": bootstrap_frame[metric].quantile(0.975),
            "valid_replicates": len(bootstrap_frame),
        }
        for metric in ["average_precision", "roc_auc", "precision", "recall", "f1"]
    ])
    display(confidence_intervals)
    print(
        "Interpretar estos resultados con la limitación de que el holdout no es "
        "completamente ciego; no reutilizarlo para seleccionar modelos o umbrales."
    )

### Interpretabilidad, errores y subgrupos

La importancia por permutación se estima sobre los folds temporales de validación de train, con hiperparámetros elegidos dentro del fold externo; así no se usa el test para decidir variables. La dependencia parcial muestra la respuesta promedio del candidato final para `lead_time` y `total_nights`. El análisis de errores revisa cancelaciones con score bajo y no cancelaciones con score alto usando las predicciones fuera de muestra de train. Las comparaciones por subgrupo son descriptivas y se omiten grupos pequeños para evitar conclusiones inestables.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

# The temporal validation windows provide genuinely out-of-fold predictions on train.
oof_validation_indices = np.concatenate([
    validation_indices for _, validation_indices in temporal_folds
])
if len(oof_validation_indices) != len(calibration_oof):
    raise AssertionError("OOF rows must align with the temporal validation windows")
oof_error_review = train_df.iloc[oof_validation_indices].copy().reset_index(drop=True)
oof_error_review["y_true"] = calibration_oof["y_true"].to_numpy(dtype=int)
oof_error_review["cancellation_probability"] = calibration_oof["y_probability"].to_numpy(dtype=float)
review_columns = [
    column for column in [
        "hotel", "lead_time", "arrival_month", "market_segment",
        "total_nights", "total_guests", "y_true", "cancellation_probability",
    ]
    if column in oof_error_review.columns
]
missed_cancellations = oof_error_review.loc[
    oof_error_review["y_true"] == 1
].nsmallest(15, "cancellation_probability")
false_alarm_risks = oof_error_review.loc[
    oof_error_review["y_true"] == 0
].nlargest(15, "cancellation_probability")
print("Cancelaciones reales con menor score (riesgo de omisión):")
display(missed_cancellations[review_columns])
print("No cancelaciones reales con mayor score (riesgo de falsa alerta):")
display(false_alarm_risks[review_columns])

# Descriptive subgroup ranking metrics; groups smaller than 200 are omitted.
subgroup_rows = []
row_offset = 0
for fold_number, (_, validation_indices) in enumerate(temporal_folds, start=1):
    fold_size = len(validation_indices)
    fold_scores = calibration_oof.iloc[row_offset:row_offset + fold_size]
    fold_data = train_df.iloc[validation_indices].reset_index(drop=True)
    row_offset += fold_size
    for group_column in ["hotel", "market_segment", "country"]:
        if group_column not in fold_data.columns:
            continue
        for group_value, group_indices in fold_data.groupby(
            group_column, dropna=False, observed=True
        ).groups.items():
            group_indices = np.asarray(list(group_indices), dtype=int)
            if len(group_indices) < 200:
                continue
            y_group = fold_scores.iloc[group_indices]["y_true"].to_numpy(dtype=int)
            score_group = fold_scores.iloc[group_indices]["y_probability"].to_numpy(dtype=float)
            if np.unique(y_group).size < 2:
                continue
            group_ap, group_capacity = capacity_scenario_metrics(y_group, score_group)
            top_ten = group_capacity.loc[
                group_capacity["fraction_alerted"] == 0.10
            ].iloc[0]
            subgroup_rows.append({
                "fold": fold_number,
                "group": group_column,
                "value": "(missing)" if pd.isna(group_value) else str(group_value),
                "rows": len(group_indices),
                "average_precision": group_ap,
                "precision_at_10_percent": top_ten["precision_at_k"],
                "recall_at_10_percent": top_ten["recall_at_k"],
            })
subgroup_metrics = pd.DataFrame(subgroup_rows)
display(subgroup_metrics.sort_values(["group", "fold", "rows"], ascending=[True, True, False]))

# Permutation importance on each temporal validation fold, not on the final test.
permutation_rows = []
for fold_number, (train_indices, validation_indices) in enumerate(temporal_folds, start=1):
    selected_candidate = nested_candidate_results.loc[
        (nested_candidate_results["outer_fold"] == fold_number)
        & (nested_candidate_results["model"] == "Gradient boosting")
        & (nested_candidate_results["selected"] == True)
    ].iloc[0]
    fold_model = clone(model_suite["Gradient boosting"])
    fold_model.set_params(**selected_candidate["parameters"])
    fold_model.fit(
        train_df.iloc[train_indices][MODEL_FEATURE_COLS],
        train_df.iloc[train_indices][TARGET].to_numpy(dtype=int),
    )
    validation_sample = train_df.iloc[validation_indices]
    if len(validation_sample) > 2500:
        validation_sample = validation_sample.sample(n=2500, random_state=SEED)
    importance = permutation_importance(
        fold_model,
        validation_sample[MODEL_FEATURE_COLS],
        validation_sample[TARGET].to_numpy(dtype=int),
        scoring="average_precision",
        n_repeats=3,
        random_state=SEED,
        n_jobs=1,
    )
    permutation_rows.extend({
        "fold": fold_number,
        "feature": feature,
        "importance": float(mean_importance),
    } for feature, mean_importance in zip(
        MODEL_FEATURE_COLS, importance.importances_mean
    ))

permutation_importance_by_fold = pd.DataFrame(permutation_rows)
permutation_importance_summary = permutation_importance_by_fold.groupby(
    "feature", as_index=False
).agg(
    importance_mean=("importance", "mean"),
    importance_std_between_folds=("importance", "std"),
)
permutation_importance_summary = permutation_importance_summary.sort_values(
    "importance_mean", ascending=False
)
display(permutation_importance_summary.head(20))
permutation_importance_summary.head(15).sort_values("importance_mean").plot.barh(
    x="feature", y="importance_mean", xerr="importance_std_between_folds",
    legend=False, figsize=(8, 6),
)
plt.xlabel("Caída de Average Precision al permutar")
plt.title("Importancia por permutación en folds temporales de train")
plt.tight_layout()
plt.show()

# Partial dependence uses a sample of train and does not consult the reserved test.
pdp_sample = train_df[MODEL_FEATURE_COLS].sample(
    n=min(2000, len(train_df)), random_state=SEED
)
PartialDependenceDisplay.from_estimator(
    final_model,
    pdp_sample,
    features=["lead_time", "total_nights"],
    response_method="predict_proba",
    target=1,
)
plt.tight_layout()
plt.show()
print(
    "Los resultados por subgrupo son descriptivos; no implican causalidad ni "
    "garantizan desempeño estable en otros hoteles o períodos."
)

## 8. Responsabilidad, riesgos y puesta en producción

**Recomendación actual:** no desplegar ni usar el score para decidir sobreventa sin una validación con datos recientes de cada hotel, costos operativos confirmados, umbral acordado y piloto supervisado. Los resultados del dataset histórico no prueban que una intervención evite cancelaciones ni que el modelo generalice a otros hoteles o períodos.

**Riesgos a evaluar:** errores de sobreventa o habitaciones vacías; deriva temporal; faltantes y cambios en canales/segmentos; desempeño y calibración distintos por hotel, país o segmento; y posible inestabilidad por duplicados y por la fecha estimada de reserva. Las tablas de subgrupos del notebook son descriptivas y no demuestran causalidad.

**Monitoreo propuesto:** revisar mensualmente distribución de variables, tasa de faltantes, tasa de cancelación observada cuando maduren las reservas, Average Precision, precision/recall en la capacidad de alertas acordada y Brier score/calibración. Investigar y considerar reentrenar si hay cambios persistentes de datos o degradación fuera de intervalos de control definidos antes del piloto. No fijar límites numéricos sin línea base operativa.

La última celda de empaquetado guarda el estimador calibrado y genera `predict.py`, que transforma una entrada nueva con las mismas variables derivadas y devuelve probabilidad y clasificación. Este artefacto debe regenerarse solo después de cerrar el umbral y ejecutar la validación final.

In [ ]:
if FINAL_THRESHOLD is None:
    print(
        "Artefactos pendientes: define FINAL_THRESHOLD tras acordar el escenario "
        "de costos con el equipo/cátedra. No se asigna un umbral arbitrario."
    )
else:
    import joblib
    from pathlib import Path

    MODEL_ARTIFACT_PATH = Path("hotel_cancellation_pipeline.joblib")
    joblib.dump(final_model, MODEL_ARTIFACT_PATH)

    predict_script_template = '''import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

MODEL_PATH = Path(__file__).with_name("hotel_cancellation_pipeline.joblib")
MODEL_FEATURES = __MODEL_FEATURES__
CANCELLATION_THRESHOLD = __FINAL_THRESHOLD__
MONTHS = {name: index for index, name in enumerate(
    ["January", "February", "March", "April", "May", "June",
     "July", "August", "September", "October", "November", "December"],
    start=1,
)}


def prepare_features(raw_data):
    data = raw_data.copy()
    if "arrival_date" not in data.columns:
        data["arrival_date"] = pd.to_datetime(dict(
            year=data["arrival_date_year"],
            month=data["arrival_date_month"].map(MONTHS),
            day=data["arrival_date_day_of_month"],
        ))
    else:
        data["arrival_date"] = pd.to_datetime(data["arrival_date"])

    data["total_nights"] = data["stays_in_weekend_nights"] + data["stays_in_week_nights"]
    data["total_guests"] = data["adults"] + data["children"].fillna(0) + data["babies"]
    data["has_children"] = (
        (data["children"].fillna(0) + data["babies"]) > 0
    ).astype(int)
    data["arrival_month"] = data["arrival_date"].dt.month
    data["arrival_dow"] = data["arrival_date"].dt.dayofweek
    data["arrival_is_weekend"] = data["arrival_dow"].isin([5, 6]).astype(int)
    data["arrival_month_sin"] = np.sin(2 * np.pi * data["arrival_month"] / 12)
    data["arrival_month_cos"] = np.cos(2 * np.pi * data["arrival_month"] / 12)
    data["lead_time_band"] = pd.cut(
        data["lead_time"],
        bins=[-1, 7, 30, 90, 180, 365, np.inf],
        labels=["0-7", "8-30", "31-90", "91-180", "181-365", "366+"],
    ).astype("object")
    prior_bookings = data["previous_cancellations"] + data["previous_bookings_not_canceled"]
    data["prev_cancel_ratio"] = data["previous_cancellations"].div(
        prior_bookings.where(prior_bookings > 0)
    ).fillna(0)
    return data


def predict_reservations(raw_data, model_path=MODEL_PATH):
    model = joblib.load(model_path)
    features = prepare_features(raw_data)
    probabilities = model.predict_proba(features[MODEL_FEATURES])[:, 1]
    result = raw_data.copy()
    result["cancellation_probability"] = probabilities
    result["predicted_cancellation"] = (
        probabilities >= CANCELLATION_THRESHOLD
    ).astype(int)
    return result


if __name__ == "__main__":
    if len(sys.argv) != 2:
        raise SystemExit("Usage: python predict.py reservations.csv")
    predictions = predict_reservations(pd.read_csv(sys.argv[1]))
    print(predictions.to_string(index=False))
'''
    predict_script = predict_script_template.replace(
        "__MODEL_FEATURES__", repr(MODEL_FEATURE_COLS)
    ).replace("__FINAL_THRESHOLD__", repr(float(FINAL_THRESHOLD)))
    Path("predict.py").write_text(predict_script, encoding="utf-8")

    example_reservation = train_df.iloc[[0]].drop(columns=[TARGET])
    example_features = add_stateless_features(example_reservation)
    example_probability = final_model.predict_proba(
        example_features[MODEL_FEATURE_COLS]
    )[:, 1]
    display(pd.DataFrame({
        "cancellation_probability": example_probability,
        "predicted_cancellation": (example_probability >= FINAL_THRESHOLD).astype(int),
    }))
    print(f"Pipeline guardado en: {MODEL_ARTIFACT_PATH}")
    print("Script de ejemplo generado en: predict.py")

## 9. Uso de IA

Registrar desde ya, para el informe: en qué tareas se usó IA y **al menos una corrección** que se hizo sobre lo que generó.

A partir del análisis realizado sobre el dataset Hotel Booking Demand, se comprobó que el problema de predicción de cancelaciones es viable desde el punto de vista de aprendizaje automático. La variable objetivo presenta un desbalance moderado (36,1% de cancelaciones en entrenamiento), por lo que se descartó utilizar accuracy como métrica principal y se propuso emplear Average Precision junto con métricas complementarias como precisión, recall, F1 y ROC-AUC.

La partición temporal se define por fecha estimada de reserva (`arrival_date - lead_time`), y el train solo conserva reservas cuya llegada ya ocurrió al corte para que su resultado estuviera disponible al entrenar. Como la fecha real de creación no existe en el dataset, esta es una aproximación que debe reconocerse como limitación.

Durante el EDA se identificaron variables con una fuerte asociación con la cancelación, especialmente lead_time, deposit_type y hotel, además de ciertos patrones estacionales relacionados con la fecha de llegada. También se detectaron variables potencialmente problemáticas desde el punto de vista de la fuga de información, como reservation_status y reservation_status_date, que fueron marcadas para ser excluidas del entrenamiento.

Se generaron nuevas variables derivadas con el objetivo de capturar información de negocio de forma más representativa, incluyendo duración total de la estadía, número total de huéspedes, presencia de niños, características temporales y métricas basadas en el historial previo del cliente. Estas variables buscan facilitar que los modelos identifiquen patrones relevantes sin incorporar información futura.

Finalmente, se definió una estrategia de modelado basada en pipelines y validación temporal, con el objetivo de garantizar reproducibilidad y evitar fuga de datos durante el preprocesamiento. Los modelos a evaluar incluirán enfoques lineales y basados en árboles, comparándose contra un baseline simple para determinar si aportan valor real al problema de negocio.

Como trabajo futuro, resta completar el entrenamiento de los modelos, realizar la selección de hiperparámetros, evaluar el desempeño en el conjunto de prueba reservado y analizar la interpretabilidad de las predicciones antes de considerar una posible implementación en producción.

In [ ]:
import pandas as pd

uso_ia = pd.DataFrame({
    "Tarea": [
        "Definición del problema",
        "EDA",
        "Features",
        "Prevención de fuga",
        "Código",
        "Informe"
    ],
    "Herramienta": [
        "ChatGPT",
        "ChatGPT",
        "ChatGPT",
        "ChatGPT",
        "ChatGPT",
        "ChatGPT"
    ],
    "Qué generó": [
        "Propuesta inicial",
        "Ideas de análisis",
        "Variables derivadas",
        "Identificación de riesgos",
        "Fragmentos Python",
        "Borradores"
    ],
    "Corrección realizada": [
        "Adaptación al caso de estudio",
        "Validación manual",
        "Selección y ajuste",
        "Revisión manual",
        "Corrección y pruebas",
        "Edición final"
    ]
})

display(uso_ia)

In [ ]:
from pathlib import Path
from docx import Document

source_docx = Path("Obligatorio Inteligencia Artificial.docx")
updated_docx = Path("Obligatorio Inteligencia Artificial - actualizado.docx")
if not source_docx.exists():
    raise FileNotFoundError(
        f"No se encontró el informe original: {source_docx}. "
        "Ejecuta esta celda desde la carpeta del proyecto."
    )

report = Document(source_docx)
report.add_page_break()
report.add_heading("Actualización de avances", level=1)
report.add_paragraph(
    "Fecha: 28 de octubre de 2026. Este anexo conserva el informe original y registra "
    "los cambios metodológicos incorporados al notebook. Los resultados se incluyen "
    "solo cuando las celdas correspondientes se ejecutaron; no se fabrican métricas."
)
report.add_heading("Cambios incorporados", level=2)
for item in [
    "Ablation de familias de variables y selección SelectFromModel dentro de validación temporal.",
    "Experimento controlado con reservation_status y reservation_status_date, comparado con el modelo honesto en los mismos folds de train.",
    "Calibración temporal fuera de muestra y escenarios hipotéticos de costos relativos; no hay costos reales ni umbral operativo acordado.",
    "Evaluación final con matriz de confusión, métricas requeridas y bootstrap, protegida para no abrir el test antes de confirmar el modelo y el umbral.",
    "Análisis de errores, subgrupos, importancia por permutación y dependencia parcial sobre validaciones temporales de train.",
    "Código para guardar el pipeline, generar predict.py y registrar las versiones instaladas en requirements.txt.",
]:
    report.add_paragraph(item, style="List Bullet")


def append_results_table(heading, frame, columns=None, max_rows=12):
    report.add_heading(heading, level=2)
    if frame is None or frame.empty:
        report.add_paragraph(
            "Pendiente de ejecución en el notebook; no hay resultados reproducibles disponibles todavía."
        )
        return
    visible = frame if columns is None else frame[[column for column in columns if column in frame.columns]]
    visible = visible.head(max_rows)
    table = report.add_table(rows=1, cols=len(visible.columns))
    table.style = "Light Shading Accent 1"
    for cell, column in zip(table.rows[0].cells, visible.columns):
        cell.text = str(column)
    for _, row in visible.iterrows():
        cells = table.add_row().cells
        for cell, value in zip(cells, row):
            if isinstance(value, (float, np.floating)):
                cell.text = f"{value:.4f}"
            else:
                cell.text = str(value)


append_results_table(
    "Ablation por familias de variables",
    globals().get("ablation_summary"),
)
append_results_table(
    "Experimento de fuga deliberada",
    globals().get("leak_summary"),
)
append_results_table(
    "Calibración temporal",
    globals().get("calibration_metrics"),
)
append_results_table(
    "Sensibilidad a costos relativos (escenarios hipotéticos)",
    globals().get("cost_sensitivity"),
    columns=[
        "scenario", "cost_fp", "cost_fn", "threshold",
        "normalized_cost_per_reservation", "alerts_per_1000",
    ],
)
append_results_table(
    "Evaluación final del test y sus intervalos",
    globals().get("confidence_intervals"),
)

report.add_heading("Limitaciones y decisiones pendientes", level=2)
for item in [
    "La fecha estimada de reserva es una aproximación y el test puede no ser completamente ciego por una inspección previa de etiquetas y filas compartidas. No seleccionar modelos ni umbrales con ese conjunto.",
    "Los costos relativos mostrados son escenarios académicos, no importes ni costos confirmados por un hotel. El equipo debe acordar el escenario con la cátedra antes de fijar el umbral.",
    "Los análisis añadidos requieren ejecutar el notebook de punta a punta. La evaluación final permanece desactivada por defecto.",
    "No recomendar despliegue sin un piloto supervisado, costos operativos, revisión de subgrupos y monitoreo de deriva y calibración.",
]:
    report.add_paragraph(item, style="List Bullet")

report.add_paragraph(
    "Las cifras previas del cuerpo del informe se conservan como estaban en el documento original. "
    "Este anexo debe actualizarse con las tablas generadas al ejecutar las nuevas celdas."
)
report.save(updated_docx)
print(f"Informe actualizado creado sin modificar el original: {updated_docx}")

In [ ]:
report = Document(updated_docx)
report.add_heading("Aclaraciones sobre el estado anterior", level=2)
for item in [
    "El cuerpo original conserva texto de una versión previa. En el notebook ya están implementados DummyClassifier, regresión logística, cuatro familias de modelos y calibración sigmoid; la calibración y los análisis agregados requieren ejecución para confirmar resultados.",
    "El experimento de fuga, ablation y selección de variables ya tienen celdas implementadas, pero sus métricas todavía no están verificadas porque el notebook no se ha ejecutado de punta a punta.",
    "La evaluación del test sigue desactivada por defecto. El anexo no reemplaza resultados previos con cifras nuevas mientras estas no se generen de forma reproducible.",
]:
    report.add_paragraph(item, style="List Bullet")
report.save(updated_docx)
print("Anexo de aclaraciones guardado en el DOCX actualizado.")